In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import time
from group_lasso import GroupLasso
from sklearn.linear_model import Lasso

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.plotting.plotting import (
    plot_overview,
    wrapper_plot_geometry,
    plot_equation,
    plot_matrices,
)
from cs_priors.simulation.mixing_model import run_simulation, just_YAX_from_simulation
from heatmap_block_performance import heatmap_average_wrong_sources
from cs_priors.solvers.vectorized_sparse_prior import sparse_prior_solution
from cs_priors.solvers.complex_lasso import complex_lasso
from cs_priors.solvers.moe_group_lasso import moe_group_lasso_solution, tensor_to_block_matrix, matrix_to_block_vector, block_vector_to_matrix, color_groups
from cs_priors.solvers.real_augmented import to_real_augmented, from_real_augmented


### Lag metoder for blokkstruktur og sammenlign med implementasjon i sparse_prior-pakken

In [ ]:

# Define the functions to make groups for group lasso
def pseudo_inverse_solution(Y_matrix, A_tensor):
    A = tensor_to_block_matrix(A_tensor)
    Y = matrix_to_block_vector(Y_matrix)
    X_pinv = block_vector_to_matrix(np.linalg.pinv(A) @ Y, A_tensor.shape[1], A_tensor.shape[2])
    return X_pinv

def no_group_solution(Y_matrix, A_tensor, max_iter=20000, alpha=1e-4):
    A = tensor_to_block_matrix(A_tensor)
    Y = matrix_to_block_vector(Y_matrix)
    num_src_x_freq = A.shape[1]
    groups = np.arange(2*num_src_x_freq)  # Each real and imaginary part is its own group

    group_reg = A.shape[0] * alpha # samples are mics x frequencies


    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_nogroup = block_vector_to_matrix(from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1)), A_tensor.shape[1], A_tensor.shape[2])
    return X_nogroup


def complex_pair_solution(Y_matrix, A_tensor, max_iter=20000, alpha=1e-4):
    A = tensor_to_block_matrix(A_tensor)
    Y = matrix_to_block_vector(Y_matrix)
    # Group real and imaginary parts together for each frequency bin
    num_src_x_freq = A.shape[1]
    groups = np.concatenate([np.arange(num_src_x_freq), np.arange(num_src_x_freq)])  # group real and imag parts of the same frequency together

    group_reg = A.shape[0] * alpha # samples are mics x frequencies


    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_cpair = block_vector_to_matrix(from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1)), A_tensor.shape[1], A_tensor.shape[2])
    return X_cpair

def pure_real_imag_solution(Y_matrix, A_tensor, max_iter=20000, alpha=1e-4):
    A = tensor_to_block_matrix(A_tensor)
    Y = matrix_to_block_vector(Y_matrix)
    num_src_x_freq = A.shape[1]
    groups = np.concatenate([np.zeros(num_src_x_freq), np.ones(num_src_x_freq)])  # group all real parts together and all imag parts together

    group_reg = A.shape[0] * alpha # samples are mics x frequencies


    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_rigroup = block_vector_to_matrix(from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1)), A_tensor.shape[1], A_tensor.shape[2])
    return X_rigroup

def frequency_group_lasso_solution(Y_matrix, A_tensor, max_iter=20000, alpha=1e-4):
    num_mics, num_sources, num_frequencies = A_tensor.shape
    A = tensor_to_block_matrix(A_tensor)
    Y = matrix_to_block_vector(Y_matrix)
    groups = np.concatenate([[i]*num_frequencies for i in range(num_sources)])
    groups = np.concatenate([groups, groups])  # for real and imaginary parts
    group_reg = num_mics * num_frequencies * alpha  # samples are mics x frequencies

    model = GroupLasso(
        groups=groups, group_reg=group_reg, n_iter=max_iter, supress_warning=True
    )
    X_freq = block_vector_to_matrix(from_real_augmented(model.fit(to_real_augmented(A), to_real_augmented(Y)).coef_.reshape(-1, 1)), num_sources, num_frequencies)
    return X_freq, groups


sim = run_simulation(seed=0, num_sources=5, num_mics=4, s_sparse=2, frequencies=[[100, 200, 300], [400, 500, 600], [100, 300], [200,400, 600], [100, 300]], angle_step=2*np.pi/5, sampling_rate_factor=3)
X0 = pseudo_inverse_solution(sim.Y, sim.C)
X_nogroup = no_group_solution(sim.Y, sim.C)
X_cpair = complex_pair_solution(sim.Y, sim.C)
X_rigroup = pure_real_imag_solution(sim.Y, sim.C)
X_freq, groups = frequency_group_lasso_solution(sim.Y, sim.C)
X_moe = moe_group_lasso_solution(sim.Y, sim.C)
group_matrix, _, _ = color_groups(groups, num_sources=sim.C.shape[1], num_frequencies=sim.C.shape[2])


plot_matrices(
    [sim.X, X0, group_matrix, X_nogroup, X_cpair, X_rigroup, X_freq, X_moe], titles=["True X Block", "Pseudo Inverse", "Group Matrix", "No Group Lasso", "Complex pair Group Lasso", "Pure Real/Imag Group Lasso", "Frequency  Group Lasso", "freq Group Lasso other file"], show_values=False
    )

# Quantify each group configuration

In [ ]:
seeds = 3
sources = 10
microphone_range= [3, 7, 9]
sparsity_range = [1,2,4]
freq_interval = (100, 1000)
max_iter = 1000
alpha = 1e-1
vmax = sources
vmin = 0

In [ ]:
# Psuedo inverse performance
# heatmap_average_wrong_sources(pseudo_inverse_solution, mic_range=microphone_range, sources=sources, sparsity_range=sparsity_range, freq_interval=freq_interval, vmin=vmin, vmax=vmax, seeds=seeds, debug=True)

In [ ]:
# No groups
heatmap_average_wrong_sources(lambda Y_matrix, A_tensor: no_group_solution(Y_matrix, A_tensor, max_iter=max_iter, alpha=alpha), method_name="No Group Lasso", mic_range=microphone_range, sources=sources, sparsity_range=sparsity_range, freq_interval=freq_interval, vmin=vmin, vmax=vmax, seeds=seeds, debug=True)

In [ ]:
# Pure real/imag groups
heatmap_average_wrong_sources(lambda Y_matrix, A_tensor: pure_real_imag_solution(Y_matrix, A_tensor, max_iter=max_iter, alpha=alpha), method_name="Pure Real/Imag Group Lasso", mic_range=microphone_range, sources=sources, sparsity_range=sparsity_range, freq_interval=freq_interval, vmin=vmin, vmax=vmax, seeds=seeds, debug=True)

In [ ]:
# Complex pair groups
heatmap_average_wrong_sources(lambda Y_matrix, A_tensor: complex_pair_solution(Y_matrix, A_tensor, max_iter=max_iter, alpha=alpha), method_name="Complex Pair Group Lasso", mic_range=microphone_range, sources=sources, sparsity_range=sparsity_range, freq_interval=freq_interval, vmin=vmin, vmax=vmax, seeds=seeds, debug=True)

In [ ]:
# Frequency groups
heatmap_average_wrong_sources(lambda Y_matrix, A_tensor: moe_group_lasso_solution(Y_matrix, A_tensor, max_iter=max_iter), method_name="Frequency Group Lasso", mic_range=microphone_range, sources=sources, sparsity_range=sparsity_range, freq_interval=freq_interval, vmin=vmin, vmax=vmax, seeds=seeds, debug=True)